# 5. Fine-tuning BERT for Canteen Food Review Classification

This notebook demonstrates how to fine-tune a BERT model to classify canteen food reviews into sentiment categories: **Positive**, **Neutral**, and **Negative**.

In [ ]:
!pip install -q transformers datasets torch evaluate scikit-learn matplotlib seaborn

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 2. Prepare the Dataset

We'll create a synthetic dataset of canteen reviews for this classification task.
Labels: 0: Negative, 1: Neutral, 2: Positive

In [ ]:
data = {
    "review": [
        "The food was absolutely delicious and the service was quick!",
        "Very disappointed. The chicken was undercooked and cold.",
        "The meal was okay, nothing special but satisfied my hunger.",
        "Found a hair in my food. Extremely unhygienic!",
        "Great value for money. The pasta is a must-try.",
        "The canteen is too noisy and crowded, but the food is decent.",
        "Terrible experience. The staff was rude and the food was salty.",
        "Best lunch I've had in a long time! Highly recommend the soup.",
        "It's an average canteen. Some days are good, some are bad.",
        "The food is getting worse every week. I'm stopping my subscription.",
        "Clean environment and tasty snacks. Love the coffee here.",
        "Waited for 30 minutes for a simple sandwich. Very slow service.",
        "The menu is quite repetitive, but the quality is consistent.",
        "Horrible taste. I couldn't even finish my plate.",
        "The variety of dishes is excellent. Always something new to try.",
        "Price is a bit high for students, but the quality is good.",
        "I like the vegetarian options they provide.",
        "The floor was slippery and the tables were not cleaned.",
        "Wonderful staff and even better food. 10/10!",
        "Not bad, but they should include more healthy options.",
        "The dessert was way too sweet, almost inedible.",
        "Perfect spot for a quick bite. Fast and efficient.",
        "I don't recommend the stir-fry. It's too oily.",
        "The biryani is fantastic. Authentic taste!",
        "Mediocre quality. Tastes like frozen food."
    ],
    "label": [2, 0, 1, 0, 2, 1, 0, 2, 1, 0, 2, 0, 1, 0, 2, 1, 2, 0, 2, 1, 0, 2, 0, 2, 1]
}

# Label mapping
id2label = {0: "NEGATIVE", 1: "NEUTRAL", 2: "POSITIVE"}
label2id = {"NEGATIVE": 0, "NEUTRAL": 1, "POSITIVE": 2}

df = pd.DataFrame(data)
print(f"Dataset size: {len(df)}")
df.head()

## 3. Preprocessing

In [ ]:
model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    return tokenizer(examples["review"], padding="max_length", truncation=True)

# Split the data: 80% train, 20% test
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Create Dataset objects
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

raw_datasets = DatasetDict({
    "train": train_dataset,
    "test": test_dataset
})

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

## 4. Model Training

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint, 
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="bert-canteen-review-classification-v2",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5, # Increased epochs for better learning on small data
    weight_decay=0.01,
    logging_steps=5,
    push_to_hub=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)

trainer.train()

## 5. Evaluation and Visualization

In [ ]:
def plot_confusion_matrix(y_true, y_pred, labels):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=labels, yticklabels=labels, cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.show()

# Get predictions on test set
predictions_output = trainer.predict(tokenized_datasets["test"])
y_pred = np.argmax(predictions_output.predictions, axis=-1)
y_true = predictions_output.label_ids

plot_confusion_matrix(y_true, y_pred, [id2label[i] for i in range(3)])
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=[id2label[i] for i in range(3)]))

## 6. Inference

Let's test the model on some new reviews.

In [ ]:
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
    model.to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    
    predicted_class_id = logits.argmax().item()
    return id2label[predicted_class_id]

# Test cases
test_reviews = [
    "The pizza was actually pretty good today.",
    "I hate standing in the long queue for such bad food.",
    "The coffee is just okay and overpriced.",
    "Delicious food and friendly staff.",
    "The canteen is closed during lunch hours. Worst management."
]

print("Predictions for new reviews:")
for review in test_reviews:
    sentiment = predict_sentiment(review)
    print(f"Review: {review}\nSentiment: {sentiment}\n")